# 04d · Score Every Transaction with the FT-Transformer (for Stacking)

Can XGBoost benefit from the FT-Transformer's *output* as an extra input feature, instead
of (or alongside) treating the two models as competitors? This is standard model stacking
— a neural net's learned representation often captures interaction patterns a single
tree-split boundary approximates only roughly, and feeding its score into a GBDT lets the
GBDT use it as just another signal, weighted (or ignored) by its own splits.

This notebook is torch-only (same reasoning as 04b/04c — see 04c's opening note on why
PyTorch and XGBoost never run in the same process here). It reproduces the exact 120k-row
training subsample notebook 04b used (`random_state=42` on the same parquet is
deterministic — no need to have separately persisted the scaler/vocabulary), loads the
saved FT-Transformer weights, and scores:

- **The training-subsample complement** (~614k rows of `train` the FT-Transformer never
  saw) — genuinely out-of-sample for the network.
- **`val`** (all of it) — 20k of these rows were looked at for early-stopping *decisions*,
  not gradient updates; a mild form of use, called out honestly rather than silently
  treated as clean.
- **`test`** — fully untouched by the FT-Transformer until its own final evaluation.

**The FT-Transformer's own 120k training rows are deliberately excluded from every score
file below.** Scoring them would give an in-sample, overfit `nn_score` — including it as a
stacking feature for those specific rows would leak the network's memorization of its own
training labels into the downstream XGBoost's training set.

In [1]:
import json
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device("cpu")

with open("../data/processed/feature_columns.json") as f:
    cfg = json.load(f)
FEATURES, LABEL = cfg["features"], cfg["label"]
CATEGORICAL_COL = "category"

train_full = pd.read_parquet("../data/processed/train.parquet")
val_full = pd.read_parquet("../data/processed/val.parquet")
test = pd.read_parquet("../data/processed/test.parquet")

# Reproduces notebook 04b's exact training subsample (same seed, same source parquet).
TRAIN_SAMPLE_SIZE = 120_000
train_sample = train_full.sample(n=TRAIN_SAMPLE_SIZE, random_state=42)
train_holdout = train_full.drop(index=train_sample.index).reset_index(drop=True)

print(f"train_full={len(train_full):,}  train_sample (excluded)={len(train_sample):,}  "
      f"train_holdout (scored)={len(train_holdout):,}")
print(f"val (scored)={len(val_full):,}  test (scored)={len(test):,}")

train_full=734,002  train_sample (excluded)=120,000  train_holdout (scored)=614,002
val (scored)=157,287  test (scored)=157,286


In [2]:
categories = sorted(train_sample[CATEGORICAL_COL].unique())
cat_to_idx = {c: i for i, c in enumerate(categories)}
UNK_IDX = len(categories)
N_CATEGORIES = len(categories) + 1
N_NUMERIC = len(FEATURES)

def cat_ids(df):
    return df[CATEGORICAL_COL].map(cat_to_idx).fillna(UNK_IDX).astype(int).values

scaler = StandardScaler()
scaler.fit(train_sample[FEATURES])  # identical fit to notebook 04b -- same subsample, same seed

print(f"{N_NUMERIC} numeric features, {len(categories)} categories + 1 unknown bucket")

21 numeric features, 14 categories + 1 unknown bucket


In [3]:
class FeatureTokenizer(nn.Module):
    def __init__(self, n_numeric, n_categories, d_token):
        super().__init__()
        self.numeric_weight = nn.Parameter(torch.empty(n_numeric, d_token))
        self.numeric_bias = nn.Parameter(torch.empty(n_numeric, d_token))
        self.cat_embedding = nn.Embedding(n_categories, d_token)
        self.cls = nn.Parameter(torch.empty(1, 1, d_token))

    def forward(self, x_num, x_cat):
        num_tokens = x_num.unsqueeze(-1) * self.numeric_weight + self.numeric_bias
        cat_tokens = self.cat_embedding(x_cat).unsqueeze(1)
        cls_tokens = self.cls.expand(x_num.size(0), -1, -1)
        return torch.cat([cls_tokens, num_tokens, cat_tokens], dim=1)


class FTTransformer(nn.Module):
    def __init__(self, n_numeric, n_categories, d_token=64, n_blocks=2, n_heads=8,
                 ffn_mult=2, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_numeric, n_categories, d_token)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * ffn_mult,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.head = nn.Sequential(
            nn.LayerNorm(d_token), nn.Linear(d_token, d_token // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_token // 2, 1),
        )

    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)
        encoded = self.encoder(tokens)
        return self.head(encoded[:, 0]).squeeze(-1)


# Same architecture hyperparameters as notebook 04b -- must match exactly to load the
# saved state dict.
D_TOKEN, N_BLOCKS, N_HEADS, DROPOUT = 64, 2, 8, 0.1
model = FTTransformer(N_NUMERIC, N_CATEGORIES, d_token=D_TOKEN, n_blocks=N_BLOCKS,
                       n_heads=N_HEADS, dropout=DROPOUT)
model.load_state_dict(torch.load("../models/ft_transformer.pt", map_location="cpu"))
model.eval()
print("Loaded FT-Transformer weights from notebook 04b")

Loaded FT-Transformer weights from notebook 04b


/var/folders/82/g56w2bw94xq30h2kr2g3npm40000gn/T/ipykernel_65027/1241544069.py:25: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)


In [4]:
@torch.no_grad()
def score_frame(df, batch_size=4096):
    x_num = torch.from_numpy(scaler.transform(df[FEATURES]).astype(np.float32))
    x_cat = torch.from_numpy(cat_ids(df).astype(np.int64))
    probs = []
    for i in range(0, len(df), batch_size):
        logits = model(x_num[i:i + batch_size], x_cat[i:i + batch_size])
        probs.append(torch.sigmoid(logits).numpy())
    return np.concatenate(probs)

nn_scores = pd.concat([
    train_holdout[["transaction_id"]].assign(
        nn_score=score_frame(train_holdout), split="train_holdout"),
    val_full[["transaction_id"]].assign(
        nn_score=score_frame(val_full), split="val"),
    test[["transaction_id"]].assign(
        nn_score=score_frame(test), split="test"),
], ignore_index=True)

nn_scores.to_parquet("../reports/nn_scores.parquet")
print(f"Saved ../reports/nn_scores.parquet  ({len(nn_scores):,} rows)")
nn_scores.groupby("split")["nn_score"].describe()

Saved ../reports/nn_scores.parquet  (928,575 rows)


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,157286.0,0.080462,0.176918,0.003346,0.004932,0.009100,0.045916,0.996813
train_holdout,614002.0,0.079616,0.185698,0.003349,0.004458,0.007587,0.035788,0.996992
val,157287.0,0.078678,0.174983,0.003359,0.004835,0.008670,0.043573,0.996977
